# Qwen3-1.7B Clinical Screening JSON Fine-Tuning

End-to-end notebook that fine-tunes `Qwen/Qwen3-1.7B-Base` with QLoRA via **Unsloth** on the Kaggle dataset `luizaaca/symptoms-to-diseases-with-reasoning`, evaluates the base versus fine-tuned model on structured JSON clinical outputs, and exports a **GGUF** artifact for local inference.

**Target hardware:** Google Colab with a Tesla T4 (16 GB VRAM).

This notebook is intentionally self-contained:
- it downloads the dataset directly from Kaggle;
- it ignores the dataset `source` column entirely;
- it uses a notebook-local Pydantic schema as the structured-output contract; and
- it aligns that schema with official LangChain structured-output patterns.

> ⚠️ This notebook is for research and screening-assistance workflows only. It does not replace professional medical judgment.

## 1. Set Up the Colab Workspace

Install the fine-tuning stack, load the main Python dependencies, confirm Kaggle credentials are available, and verify the GPU runtime before downloading the dataset.

In [ ]:
%%capture
%pip install --upgrade --no-cache-dir unsloth
%pip install --no-deps unsloth_zoo
%pip install bitsandbytes trl peft accelerate datasets kagglehub evaluate bert_score scikit-learn matplotlib seaborn pandas sentencepiece protobuf pydantic "langchain>=1.2"

In [ ]:
import gc
import json
import os
import random
import re
import warnings
from pathlib import Path
from textwrap import dedent
from typing import Any, cast
from google.colab import userdata

import evaluate
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from IPython.display import display
from pydantic import BaseModel, Field, ValidationError
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

CONFIG = {
    "dataset_handle": "luizaaca/symptoms-to-diseases-with-reasoning",
    "dataset_filename": "combined_diseases_symptoms_2_enriched_with_exams_v2.csv",
    "base_model_id": "Qwen/Qwen3-1.7B-Base",
    "max_seq_length": 1024,
    "load_in_4bit": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "warmup_steps": 20,
    "max_steps": 400,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "fp16": True,
    "bf16": False,
    "optim": "adamw_8bit",
    "lr_scheduler_type": "linear",
    "test_size": 0.10,
    "eval_max_samples": 300,
    "qualitative_n": 8,
    "bertscore_model_type": "distilbert-base-uncased",
    "output_dir": "outputs_qwen3_1_7b_json",
    "lora_dir": "qwen3-1.7b-clinical-json-lora",
    "gguf_dir": "qwen3-1.7b-clinical-json-gguf",
    "valid_support_statuses": ["supported", "inconclusive"],
    "seed": SEED,
}

REQUIRED_COLUMNS = [
    "input",
    "output",
    "support_status",
    "candidate_diseases",
    "recommended_exams_tests",
]

kaggle_username = userdata.get("KAGGLE_USERNAME")
kaggle_key = userdata.get("KAGGLE_KEY")
if not kaggle_username or not kaggle_key:
    print(
        "Kaggle credentials not found in the environment. "
        "In Colab, set KAGGLE_USERNAME and KAGGLE_KEY before downloading the dataset."
    )
else:
    print(f"Kaggle credentials detected for user: {kaggle_username}")

assert torch.cuda.is_available(), "CUDA GPU is required (Google Colab T4 recommended)."
gpu_name = torch.cuda.get_device_name(0)
total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}")
print(f"Total VRAM: {total_mem_gb:.1f} GB")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
print("Configuration loaded.")

## 2. Download the Kaggle Dataset

Use the Kaggle dataset `luizaaca/symptoms-to-diseases-with-reasoning` as the single training source for this notebook. The dataset page currently reports one CSV file named `combined_diseases_symptoms_2_enriched_with_exams_v2.csv` under a **CC BY-SA 4.0** license, so keep attribution and share-alike obligations in mind if you redistribute derived assets.

In [ ]:
def find_dataset_csv(root_dir: str, expected_filename: str) -> str:
    """Return the expected CSV path inside a downloaded Kaggle dataset directory.

    Args:
        root_dir: Root directory returned by `kagglehub.dataset_download`.
        expected_filename: Preferred CSV filename to locate.

    Returns:
        Absolute path to the selected CSV file.

    Raises:
        FileNotFoundError: If the dataset directory does not contain any CSV files.
    """

    csv_paths = sorted(Path(root_dir).rglob("*.csv"))
    if not csv_paths:
        raise FileNotFoundError(f"No CSV files found under {root_dir}")

    for csv_path in csv_paths:
        if csv_path.name == expected_filename:
            return str(csv_path)
    return str(csv_paths[0])


dataset_root = kagglehub.dataset_download(CONFIG["dataset_handle"])
DATASET_CSV = find_dataset_csv(dataset_root, CONFIG["dataset_filename"])

print(f"Dataset root: {dataset_root}")
print(f"Resolved CSV: {DATASET_CSV}")

## 3. Canonicalize the Structured Output Contract

The fine-tuned model will learn to emit a single JSON object with three fields:
- `support_status`
- `candidate_diseases`
- `recommended_exams_tests`

The main benchmark will use the ordered disease candidates for scoring across all held-out rows. The `support_status` field is still trained and reported, but it is evaluated in its own separate report rather than acting as a benchmark filter.

In [ ]:
class ClinicalScreeningOutput(BaseModel):
    """Structured clinical screening payload aligned with official LangChain schemas.

    Attributes:
        support_status: Free-text support status emitted by the model.
        candidate_diseases: Ordered disease candidates with the primary label first.
        recommended_exams_tests: Ordered confirmatory exams or tests.
    """

    support_status: str = Field(
        description="Support status string. Expected training values are 'supported' or 'inconclusive'.",
    )
    candidate_diseases: list[str] = Field(
        min_length=1,
        description="Ordered candidate diseases with the primary candidate in the first position.",
    )
    recommended_exams_tests: list[str] = Field(
        min_length=1,
        description="Ordered confirmatory exams or tests.",
    )


def clean_text(value: Any) -> str:
    """Return a whitespace-normalized string representation of a value.

    Args:
        value: Arbitrary value to normalize.

    Returns:
        A trimmed string with internal whitespace collapsed to single spaces.
    """

    return re.sub(r"\s+", " ", str(value)).strip()



def normalize_label(value: Any) -> str:
    """Return a normalized disease label for comparisons.

    Args:
        value: Raw disease label.

    Returns:
        Lowercase disease label with collapsed whitespace.
    """

    return clean_text(value).lower()



def unique_preserve(items: list[str]) -> list[str]:
    """Return a list with duplicates removed while preserving the first occurrence.

    Args:
        items: Ordered string values.

    Returns:
        Deduplicated ordered list.
    """

    seen: set[str] = set()
    ordered: list[str] = []
    for item in items:
        cleaned = clean_text(item)
        if not cleaned:
            continue
        key = cleaned.lower()
        if key in seen:
            continue
        seen.add(key)
        ordered.append(cleaned)
    return ordered



def parse_json_list(raw_value: Any, field_name: str) -> list[str]:
    """Parse a JSON-encoded list field from the dataset.

    Args:
        raw_value: Raw CSV value or already-materialized list.
        field_name: Column name used for clearer error messages.

    Returns:
        Parsed list of strings.

    Raises:
        ValueError: If the value cannot be parsed into a non-string list.
    """

    parsed_value = raw_value
    if isinstance(raw_value, str):
        parsed_value = json.loads(raw_value)
    if not isinstance(parsed_value, list):
        raise ValueError(f"{field_name} must decode to a list.")
    return [clean_text(item) for item in parsed_value if clean_text(item)]



def canonicalize_support_status(raw_value: Any) -> str:
    """Normalize a support-status value without enforcing it for the main benchmark.

    Args:
        raw_value: Raw support-status value.

    Returns:
        Lowercase normalized support status.
    """

    return normalize_label(raw_value)



def canonicalize_output_row(row: dict[str, Any]) -> ClinicalScreeningOutput:
    """Canonicalize one dataset row into the structured target schema.

    Args:
        row: Mapping with dataset columns.

    Returns:
        Canonical structured payload.

    Raises:
        ValueError: If required JSON-list fields are empty after normalization.
    """

    target_label = normalize_label(row["output"])
    candidate_diseases = [
        normalize_label(item) for item in parse_json_list(row["candidate_diseases"], "candidate_diseases")
    ]
    candidate_diseases = unique_preserve([target_label, *candidate_diseases])
    if not candidate_diseases:
        raise ValueError("candidate_diseases cannot be empty after normalization.")

    recommended_exams_tests = unique_preserve(
        parse_json_list(row["recommended_exams_tests"], "recommended_exams_tests")
    )
    if not recommended_exams_tests:
        raise ValueError("recommended_exams_tests cannot be empty after normalization.")

    return ClinicalScreeningOutput(
        support_status=canonicalize_support_status(row["support_status"]),
        candidate_diseases=candidate_diseases,
        recommended_exams_tests=recommended_exams_tests,
    )



def dump_canonical_json(payload: ClinicalScreeningOutput) -> str:
    """Serialize a canonical payload into a stable compact JSON string.

    Args:
        payload: Structured payload to serialize.

    Returns:
        Compact UTF-8 JSON string with stable field order.
    """

    return json.dumps(payload.model_dump(), ensure_ascii=False, separators=(",", ":"))



def structured_summary(payload: dict[str, Any]) -> str:
    """Render a structured payload into a stable plain-text summary.

    Args:
        payload: Structured payload dictionary.

    Returns:
        Canonical text summary suitable for semantic metrics such as BERTScore.
    """

    support_status = canonicalize_support_status(payload.get("support_status", "<missing>")) or "<missing>"
    candidate_diseases = ", ".join(
        normalize_label(item) for item in payload.get("candidate_diseases", []) if clean_text(item)
    ) or "<none>"
    recommended_exams = ", ".join(
        clean_text(item) for item in payload.get("recommended_exams_tests", []) if clean_text(item)
    ) or "<none>"
    return (
        f"support_status: {support_status}; "
        f"candidate_diseases: {candidate_diseases}; "
        f"recommended_exams_tests: {recommended_exams}"
    )

In [ ]:
def stratified_split(
    dataframe: pd.DataFrame,
    test_size: float,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split a dataframe into train and test partitions with label stratification.

    Args:
        dataframe: Input dataframe with a `normalized_output` column.
        test_size: Test-set ratio.
        seed: Random seed used by `train_test_split`.

    Returns:
        Tuple of `(train_df, test_df)`.
    """

    train_df, test_df = train_test_split(
        dataframe,
        test_size=test_size,
        random_state=seed,
        stratify=dataframe["normalized_output"],
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)



def stratified_eval_sample(dataframe: pd.DataFrame, cap: int) -> pd.DataFrame:
    """Return a capped evaluation subset while preserving class balance as much as possible.

    Args:
        dataframe: Held-out dataframe with a `normalized_output` column.
        cap: Maximum number of rows to keep.

    Returns:
        Stratified evaluation subset.
    """

    if len(dataframe) <= cap:
        return dataframe.reset_index(drop=True)

    shuffled = dataframe.sample(frac=1.0, random_state=SEED)
    per_class = max(1, cap // shuffled["normalized_output"].nunique())
    sampled = shuffled.groupby("normalized_output", group_keys=False).head(per_class)
    if len(sampled) < cap:
        remainder = shuffled.drop(index=sampled.index)
        sampled = pd.concat([sampled, remainder.head(cap - len(sampled))], ignore_index=True)
    elif len(sampled) > cap:
        sampled = sampled.head(cap)
    return sampled.reset_index(drop=True)


raw_df = pd.read_csv(DATASET_CSV)
missing_columns = [column for column in REQUIRED_COLUMNS if column not in raw_df.columns]
assert not missing_columns, f"Missing required columns: {missing_columns}"

working_df = raw_df.loc[:, REQUIRED_COLUMNS].copy()
canonical_payloads: list[ClinicalScreeningOutput | None] = []
canonical_json_targets: list[str] = []
preprocessing_errors: list[str] = []

for row in working_df.itertuples(index=False):
    row_mapping = row._asdict()
    try:
        payload = canonicalize_output_row(cast(dict[str, Any], row_mapping))
        canonical_payloads.append(payload)
        canonical_json_targets.append(dump_canonical_json(payload))
        preprocessing_errors.append("")
    except Exception as exc:
        canonical_payloads.append(None)
        canonical_json_targets.append("")
        preprocessing_errors.append(f"{type(exc).__name__}: {exc}")

valid_mask = np.array([payload is not None for payload in canonical_payloads], dtype=bool)
valid_payloads = [payload for payload in canonical_payloads if payload is not None]
clean_df = working_df.loc[valid_mask].copy().reset_index(drop=True)
clean_df["structured_target"] = pd.Series(
    [payload.model_dump() for payload in valid_payloads],
    dtype="object",
)
clean_df["structured_target_json"] = [dump_canonical_json(payload) for payload in valid_payloads]
clean_df["normalized_output"] = clean_df["output"].map(normalize_label)
clean_df["reference_support_status"] = [payload.support_status for payload in valid_payloads]
clean_df["reference_support_status_valid"] = clean_df["reference_support_status"].isin(CONFIG["valid_support_statuses"])

label_counts = clean_df["normalized_output"].value_counts()
kept_labels = label_counts[label_counts >= 2].index
model_df = clean_df[clean_df["normalized_output"].isin(kept_labels)].reset_index(drop=True)
rare_label_rows_dropped = int(len(clean_df) - len(model_df))

train_df, test_df = stratified_split(model_df, CONFIG["test_size"], SEED)
eval_df = stratified_eval_sample(test_df, CONFIG["eval_max_samples"])
qualitative_df = eval_df.sample(n=min(CONFIG["qualitative_n"], len(eval_df)), random_state=SEED).reset_index(drop=True)

preprocessing_summary = pd.DataFrame(
    [
        {"metric": "raw_rows", "value": int(len(raw_df))},
        {"metric": "rows_after_required_column_selection", "value": int(len(working_df))},
        {"metric": "rows_with_valid_structured_targets", "value": int(len(clean_df))},
        {"metric": "rows_with_preprocessing_errors", "value": int((~valid_mask).sum())},
        {"metric": "rows_dropped_for_rare_labels", "value": rare_label_rows_dropped},
        {"metric": "rows_used_for_training_and_eval", "value": int(len(model_df))},
        {"metric": "unique_normalized_labels", "value": int(model_df["normalized_output"].nunique())},
        {
            "metric": "reference_support_status_valid_rate",
            "value": float(clean_df["reference_support_status_valid"].mean()),
        },
    ]
)

display(preprocessing_summary)
if (~valid_mask).any():
    invalid_examples = working_df.loc[~valid_mask, ["input", "output"]].head(5).copy()
    invalid_examples["error"] = pd.Series(
        [error for error in preprocessing_errors if error][: len(invalid_examples)],
        dtype="object",
    )
    display(invalid_examples)

split_summary = pd.DataFrame(
    [
        {"split": "train", "rows": int(len(train_df)), "labels": int(train_df["normalized_output"].nunique())},
        {"split": "test", "rows": int(len(test_df)), "labels": int(test_df["normalized_output"].nunique())},
        {"split": "eval", "rows": int(len(eval_df)), "labels": int(eval_df["normalized_output"].nunique())},
        {"split": "qualitative", "rows": int(len(qualitative_df)), "labels": int(qualitative_df["normalized_output"].nunique())},
    ]
)

display(split_summary)
display(model_df[["input", "output", "structured_target_json"]].head(3))

## 4. Align the JSON Contract with Official LangChain Patterns

Official LangChain structured-output guidance recommends schema-driven responses via `response_format=Schema` or `with_structured_output(Schema)`. This notebook keeps the schema notebook-local and trains the model to emit JSON that can be validated directly against that same Pydantic model downstream.

In [ ]:
from langchain.agents import create_agent

SCHEMA_JSON = json.dumps(ClinicalScreeningOutput.model_json_schema(), ensure_ascii=False, indent=2)
USER_INSTRUCTION = (
    "Analyze the reported symptoms and return only the structured clinical screening JSON object."
)
SYSTEM_PROMPT = dedent(
    f"""
    You are a clinical screening model.
    Return exactly one valid JSON object and nothing else.

    The JSON object must match this schema:
    {SCHEMA_JSON}

    Rules:
    - Do not add markdown fences.
    - Do not add explanations before or after the JSON.
    - Keep `candidate_diseases` ordered with the primary candidate first.
    - Keep `recommended_exams_tests` as a non-empty list.
    - Prefer `support_status` values such as `supported` or `inconclusive`.
    """
).strip()

LANGCHAIN_AGENT_EXAMPLE = dedent(
    """
    from langchain.agents import create_agent

    agent = create_agent(
        model="your-chat-model",
        response_format=ClinicalScreeningOutput,
    )
    """
).strip()

LANGCHAIN_MODEL_EXAMPLE = dedent(
    """
    model_with_structure = chat_model.with_structured_output(ClinicalScreeningOutput)
    result = model_with_structure.invoke("Reported symptoms: fever, myalgia, headache")
    """
).strip()

print(f"Official LangChain helper available: {create_agent.__name__}")
print("\nSchema preview:\n")
print(SCHEMA_JSON[:1200])
print("\nLangChain response_format example:\n")
print(LANGCHAIN_AGENT_EXAMPLE)
print("\nLangChain with_structured_output example:\n")
print(LANGCHAIN_MODEL_EXAMPLE)

## 5. Fine-Tune Qwen3-1.7B on Canonical JSON Targets

Load the base model in 4-bit, attach LoRA adapters, render Qwen chat-format examples with JSON-only assistant targets, and fine-tune with response-only supervision.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["base_model_id"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

model.print_trainable_parameters()

In [ ]:
def build_user_message(symptoms: str) -> str:
    """Build the user turn for one training or inference example.

    Args:
        symptoms: Raw symptom text.

    Returns:
        User-facing prompt content.
    """

    return (
        f"{USER_INSTRUCTION}\n\n"
        f"Reported symptoms:\n{clean_text(symptoms)}\n\n"
        "Return exactly one JSON object that matches the schema."
    )



def to_chat_record(row: Any) -> dict[str, Any]:
    """Convert a dataframe row into a Qwen chat record.

    Args:
        row: Dataset row containing input symptoms and canonical target JSON.

    Returns:
        Dictionary with chat messages and the serialized assistant target.
    """

    input_text = clean_text(row["input"])
    target_json = str(row["structured_target_json"])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_message(input_text)},
        {"role": "assistant", "content": target_json},
    ]
    return {
        "messages": messages,
        "structured_target_json": target_json,
    }



def build_text_dataset(dataframe: pd.DataFrame) -> Dataset:
    """Render a dataframe into a chat-templated training dataset.

    Args:
        dataframe: Training dataframe with canonical JSON targets.

    Returns:
        Hugging Face dataset containing rendered `text` prompts.
    """

    records = [to_chat_record(record) for record in dataframe.to_dict(orient="records")]
    dataset = Dataset.from_list(records)
    dataset = dataset.map(
        lambda example: {
            "text": tokenizer.apply_chat_template(
                example["messages"],
                tokenize=False,
                add_generation_prompt=False,
            )
        }
    )
    return dataset


train_ds = build_text_dataset(train_df)
assert len(train_ds) == len(train_df), "Training dataset row count mismatch."
assert train_df["structured_target_json"].str.startswith("{").all(), "All targets must be JSON objects."

print(f"Training rows: {len(train_ds):,}")
print("\n--- Sample rendered training example ---\n")
print(train_ds[0]["text"][:1600])

In [ ]:
sft_config = SFTConfig(
    output_dir=CONFIG["output_dir"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    warmup_steps=CONFIG["warmup_steps"],
    max_steps=CONFIG["max_steps"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    logging_steps=CONFIG["logging_steps"],
    fp16=CONFIG["fp16"],
    bf16=CONFIG["bf16"],
    optim=CONFIG["optim"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    seed=SEED,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=sft_config,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("Trainer initialized.")

In [ ]:
train_result = trainer.train()
print(train_result)

log_history = pd.DataFrame(trainer.state.log_history)
loss_df = log_history.dropna(subset=["loss"])[["step", "loss"]].reset_index(drop=True)
assert len(loss_df) >= 2, "Expected at least two logged loss points during training."
assert loss_df["loss"].iloc[-1] < loss_df["loss"].iloc[0], "Final loss must be lower than initial loss."

_, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_df["step"], loss_df["loss"], marker="o", linewidth=1.5)
ax.set_xlabel("Step")
ax.set_ylabel("Training loss")
ax.set_title("Qwen3-1.7B QLoRA — training loss")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Benchmark the Base and Fine-Tuned Models

Keep the current notebook's benchmark family, but run the main benchmark across all held-out evaluation rows. The primary label is taken from the first predicted item in `candidate_diseases`. Support-status behavior is summarized separately later.

In [ ]:
from tqdm.auto import tqdm



def cleanup_memory() -> None:
    """Release Python and CUDA memory before large inference steps."""

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



def build_generation_prompt(symptoms: str) -> str:
    """Render a single inference prompt for the current schema contract.

    Args:
        symptoms: Symptom text for the current example.

    Returns:
        Chat-formatted prompt text with a generation marker.
    """

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_message(symptoms)},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )



@torch.inference_mode()
def generate_raw_output(
    mdl: Any,
    symptoms: str,
    max_new_tokens: int = 256,
) -> str:
    """Generate one deterministic model response for the supplied symptoms.

    Args:
        mdl: Loaded causal language model.
        symptoms: Symptom text.
        max_new_tokens: Maximum generation length.

    Returns:
        Raw decoded model text.
    """

    prompt = build_generation_prompt(symptoms)
    model_inputs = tokenizer(prompt, return_tensors="pt").to(mdl.device)
    output_ids = mdl.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output_ids[0][model_inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()



def extract_json_candidate(raw_text: str) -> tuple[str | None, bool]:
    """Extract the most likely JSON object from a raw model response.

    Args:
        raw_text: Raw generated text.

    Returns:
        Tuple of `(json_text, strict_json)` where `strict_json` is true only when
        the raw stripped response is already a valid JSON object.
    """

    stripped = raw_text.strip()
    if not stripped:
        return None, False

    try:
        parsed = json.loads(stripped)
        if isinstance(parsed, dict):
            return stripped, True
    except json.JSONDecodeError:
        pass

    fenced_match = re.search(r"```(?:json)?\s*(.*?)\s*```", stripped, flags=re.DOTALL | re.IGNORECASE)
    if fenced_match:
        fenced_candidate = fenced_match.group(1).strip()
        try:
            parsed = json.loads(fenced_candidate)
            if isinstance(parsed, dict):
                return fenced_candidate, False
        except json.JSONDecodeError:
            pass

    decoder = json.JSONDecoder()
    for start_index, character in enumerate(stripped):
        if character != "{":
            continue
        try:
            parsed, end_index = decoder.raw_decode(stripped[start_index:])
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            return stripped[start_index : start_index + end_index], False
    return None, False



def parse_prediction(raw_text: str) -> dict[str, Any]:
    """Parse one raw model response into benchmark-ready structured fields.

    Args:
        raw_text: Raw generated model response.

    Returns:
        Parsed benchmark dictionary with JSON validity, schema validity, and
        projected primary label fields.
    """

    json_candidate, strict_json = extract_json_candidate(raw_text)
    parsed_payload: dict[str, Any] = {}
    json_parsed = False
    schema_valid = False
    support_status = ""
    candidate_diseases: list[str] = []
    recommended_exams_tests: list[str] = []

    if json_candidate is not None:
        loaded_candidate = json.loads(json_candidate)
        if isinstance(loaded_candidate, dict):
            parsed_payload = loaded_candidate
            json_parsed = True
            raw_candidates = loaded_candidate.get("candidate_diseases", [])
            raw_exams = loaded_candidate.get("recommended_exams_tests", [])
            candidate_diseases = unique_preserve(
                [normalize_label(item) for item in raw_candidates] if isinstance(raw_candidates, list) else []
            )
            recommended_exams_tests = unique_preserve(
                [clean_text(item) for item in raw_exams] if isinstance(raw_exams, list) else []
            )
            support_status = canonicalize_support_status(loaded_candidate.get("support_status", ""))
            try:
                ClinicalScreeningOutput(
                    support_status=support_status,
                    candidate_diseases=candidate_diseases,
                    recommended_exams_tests=recommended_exams_tests,
                )
                schema_valid = bool(support_status)
            except ValidationError:
                schema_valid = False

    summary_payload = {
        "support_status": support_status,
        "candidate_diseases": candidate_diseases,
        "recommended_exams_tests": recommended_exams_tests,
    }
    return {
        "raw_output": raw_text,
        "json_candidate": json_candidate or "",
        "strict_json": strict_json,
        "json_parsed": json_parsed,
        "schema_valid": schema_valid,
        "support_status": support_status,
        "support_status_valid": support_status in CONFIG["valid_support_statuses"],
        "candidate_diseases": candidate_diseases,
        "recommended_exams_tests": recommended_exams_tests,
        "predicted_label": candidate_diseases[0] if candidate_diseases else "",
        "summary_text": structured_summary(summary_payload),
        "parsed_payload": parsed_payload,
    }



def predict_dataframe(mdl: Any, dataframe: pd.DataFrame, desc: str) -> pd.DataFrame:
    """Run deterministic inference over a dataframe and parse every prediction.

    Args:
        mdl: Loaded model to evaluate.
        dataframe: Evaluation dataframe.
        desc: Progress-bar label.

    Returns:
        Dataframe containing reference columns and parsed predictions.
    """

    rows: list[dict[str, Any]] = []
    for row in tqdm(dataframe.itertuples(index=False), total=len(dataframe), desc=desc):
        raw_output = generate_raw_output(mdl, str(row.input))
        reference_payload = cast(dict[str, Any], row.structured_target)
        parsed = parse_prediction(raw_output)
        rows.append(
            {
                "input": str(row.input),
                "reference_output": str(row.output),
                "reference_norm": str(row.normalized_output),
                "reference_support_status": str(row.reference_support_status),
                "reference_support_status_valid": bool(row.reference_support_status_valid),
                "reference_payload": reference_payload,
                "reference_summary": structured_summary(reference_payload),
                **parsed,
            }
        )
    return pd.DataFrame(rows)



def compute_label_metrics(predictions_df: pd.DataFrame) -> dict[str, Any]:
    """Compute primary-label and JSON-structure metrics for one prediction dataframe.

    Args:
        predictions_df: Parsed prediction dataframe.

    Returns:
        Dictionary with label and structure metrics.
    """

    y_true = predictions_df["reference_norm"].tolist()
    y_pred = [normalize_label(label) for label in predictions_df["predicted_label"].tolist()]
    target_in_candidates = predictions_df.apply(
        lambda row: row["reference_norm"] in row["candidate_diseases"],
        axis=1,
    )
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred)),
        "strict_json_rate": float(predictions_df["strict_json"].mean()),
        "json_parse_rate": float(predictions_df["json_parsed"].mean()),
        "schema_valid_rate": float(predictions_df["schema_valid"].mean()),
        "target_label_in_candidates_rate": float(target_in_candidates.mean()),
        "non_empty_exam_list_rate": float(predictions_df["recommended_exams_tests"].map(bool).mean()),
        "n": int(len(predictions_df)),
    }



def plot_confusion(
    y_true: list[str],
    y_pred: list[str],
    title: str,
    max_classes: int = 24,
    annot_threshold: float = 0.05,
) -> None:
    """Plot a row-normalized confusion matrix for primary-label predictions.

    Args:
        y_true: Reference labels.
        y_pred: Predicted labels.
        title: Plot title.
        max_classes: Maximum number of plotted classes.
        annot_threshold: Minimum cell value to annotate.
    """

    normalized_true = [normalize_label(label) for label in y_true]
    normalized_pred = [normalize_label(label) or "<no_label>" for label in y_pred]
    labels = sorted(set(normalized_true) | set(normalized_pred))
    matrix = confusion_matrix(normalized_true, normalized_pred, labels=labels, normalize="true")

    if len(labels) > max_classes:
        off_diagonal_mass = matrix.sum(axis=1) - np.diag(matrix)
        top_indices = np.argsort(off_diagonal_mass)[::-1][:max_classes]
        keep = sorted(top_indices.tolist())
        labels = [labels[index] for index in keep]
        matrix = matrix[np.ix_(keep, keep)]
        row_sums = matrix.sum(axis=1, keepdims=True)
        matrix = np.divide(matrix, row_sums, out=np.zeros_like(matrix), where=row_sums > 0)
        title = f"{title} (top-{max_classes} most-confused classes)"

    annotations = np.where(matrix >= annot_threshold, np.vectorize(lambda value: f"{value:.0%}")(matrix), "")
    _, ax = plt.subplots(figsize=(max(8, len(labels) * 0.45), max(7, len(labels) * 0.45)))
    sns.heatmap(
        matrix,
        annot=annotations,
        fmt="",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        cbar_kws={"label": "Recall (row-normalized)"},
        linewidths=0.3,
        linecolor="white",
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


FastLanguageModel.for_inference(model)
print("Inference helpers ready.")

In [ ]:
with model.disable_adapter():
    qualitative_base = predict_dataframe(model, qualitative_df, "Base qualitative")
qualitative_fine_tuned = predict_dataframe(model, qualitative_df, "Fine-tuned qualitative")

qualitative_results = pd.DataFrame(
    {
        "symptoms": qualitative_df["input"].str.slice(0, 120).values,
        "reference_label": qualitative_df["output"].values,
        "reference_json": qualitative_df["structured_target_json"].values,
        "base_raw": qualitative_base["raw_output"].values,
        "base_primary_label": qualitative_base["predicted_label"].values,
        "fine_tuned_raw": qualitative_fine_tuned["raw_output"].values,
        "fine_tuned_primary_label": qualitative_fine_tuned["predicted_label"].values,
    }
)

with pd.option_context("display.max_colwidth", None):
    display(qualitative_results)

In [ ]:
fine_tuned_predictions = predict_dataframe(model, eval_df, "Fine-tuned eval")
with model.disable_adapter():
    base_predictions = predict_dataframe(model, eval_df, "Base eval")

metrics_rows = []
for model_name, predictions_df in [("Base", base_predictions), ("Fine-tuned", fine_tuned_predictions)]:
    metrics = compute_label_metrics(predictions_df)
    metrics["model"] = model_name
    metrics_rows.append(metrics)

metrics_df = pd.DataFrame(metrics_rows)[
    [
        "model",
        "n",
        "accuracy",
        "f1_macro",
        "cohen_kappa",
        "strict_json_rate",
        "json_parse_rate",
        "schema_valid_rate",
        "target_label_in_candidates_rate",
        "non_empty_exam_list_rate",
    ]
].sort_values("model")

display(metrics_df)

fine_tuned_accuracy = metrics_df.loc[metrics_df["model"] == "Fine-tuned", "accuracy"].iloc[0]
base_accuracy = metrics_df.loc[metrics_df["model"] == "Base", "accuracy"].iloc[0]
print({"fine_tuned_accuracy": fine_tuned_accuracy, "base_accuracy": base_accuracy})
assert fine_tuned_accuracy >= base_accuracy, "Fine-tuned label accuracy should match or exceed the base model."

In [ ]:
plot_confusion(
    eval_df["normalized_output"].tolist(),
    fine_tuned_predictions["predicted_label"].tolist(),
    "Fine-tuned primary-label confusion matrix",
)

bertscore = evaluate.load("bertscore")


def run_bertscore(reference_summaries: list[str], predicted_summaries: list[str]) -> dict[str, Any]:
    """Compute BERTScore over canonical structured summaries.

    Args:
        reference_summaries: Reference structured summaries.
        predicted_summaries: Predicted structured summaries.

    Returns:
        Aggregate and per-sample BERTScore values.
    """

    result = bertscore.compute(
        predictions=predicted_summaries,
        references=reference_summaries,
        model_type=CONFIG["bertscore_model_type"],
        lang="en",
    )
    return {
        "precision_mean": float(np.mean(result["precision"])),
        "recall_mean": float(np.mean(result["recall"])),
        "f1_mean": float(np.mean(result["f1"])),
        "f1_std": float(np.std(result["f1"])),
        "f1_per_sample": result["f1"],
    }


reference_summaries = fine_tuned_predictions["reference_summary"].tolist()
bertscore_results = {
    "Base": run_bertscore(reference_summaries, base_predictions["summary_text"].tolist()),
    "Fine-tuned": run_bertscore(reference_summaries, fine_tuned_predictions["summary_text"].tolist()),
}

bertscore_summary = pd.DataFrame(
    [
        {
            "model": model_name,
            "BERT-P": metrics["precision_mean"],
            "BERT-R": metrics["recall_mean"],
            "BERT-F1": metrics["f1_mean"],
            "BERT-F1 std": metrics["f1_std"],
        }
        for model_name, metrics in bertscore_results.items()
    ]
).sort_values("model")

display(bertscore_summary)

_, ax = plt.subplots(figsize=(8, 4))
for model_name in ["Base", "Fine-tuned"]:
    ax.hist(
        bertscore_results[model_name]["f1_per_sample"],
        bins=30,
        alpha=0.55,
        label=model_name,
    )
ax.set_title("BERTScore F1 distribution on structured summaries")
ax.set_xlabel("BERTScore F1")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def build_support_status_report(predictions_df: pd.DataFrame, model_name: str) -> dict[str, Any]:
    """Build a support-status report that is separate from the main benchmark.

    Args:
        predictions_df: Parsed predictions for one model.
        model_name: Human-readable model name.

    Returns:
        Dictionary with status-level accuracy and distribution metrics.
    """

    reference_status = predictions_df["reference_support_status"].tolist()
    predicted_status = [status if status else "<missing>" for status in predictions_df["support_status"].tolist()]
    return {
        "model": model_name,
        "support_status_accuracy": accuracy_score(reference_status, predicted_status),
        "support_status_valid_rate": float(predictions_df["support_status_valid"].mean()),
        "n": int(len(predictions_df)),
        "reference_distribution": pd.Series(reference_status).value_counts(normalize=True).to_dict(),
        "predicted_distribution": pd.Series(predicted_status).value_counts(normalize=True).to_dict(),
    }


support_status_report_df = pd.DataFrame(
    [
        build_support_status_report(base_predictions, "Base"),
        build_support_status_report(fine_tuned_predictions, "Fine-tuned"),
    ]
)

display(support_status_report_df[["model", "n", "support_status_accuracy", "support_status_valid_rate"]])

fine_tuned_status_labels = sorted(
    set(fine_tuned_predictions["reference_support_status"].tolist())
    | set([status if status else "<missing>" for status in fine_tuned_predictions["support_status"].tolist()])
)
status_matrix = confusion_matrix(
    fine_tuned_predictions["reference_support_status"].tolist(),
    [status if status else "<missing>" for status in fine_tuned_predictions["support_status"].tolist()],
    labels=fine_tuned_status_labels,
    normalize="true",
)

_, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    status_matrix,
    annot=np.where(status_matrix > 0, np.vectorize(lambda value: f"{value:.0%}")(status_matrix), ""),
    fmt="",
    cmap="Purples",
    xticklabels=fine_tuned_status_labels,
    yticklabels=fine_tuned_status_labels,
    cbar_kws={"label": "Recall (row-normalized)"},
    ax=ax,
)
ax.set_title("Fine-tuned support-status report")
ax.set_xlabel("Predicted support_status")
ax.set_ylabel("Reference support_status")
plt.tight_layout()
plt.show()

support_status_errors = fine_tuned_predictions.loc[
    fine_tuned_predictions["reference_support_status"] != fine_tuned_predictions["support_status"],
    ["input", "reference_support_status", "support_status", "raw_output"],
].head(10)

with pd.option_context("display.max_colwidth", 160):
    display(support_status_errors)

## 7. Export the Adapter and GGUF Artifact

Save the LoRA adapter, export a `q4_k_m` GGUF file, and run a smoke test that verifies the exported model still returns valid JSON for the notebook-local schema.

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/screening_robot")
except ImportError:
    ARTIFACT_ROOT = Path("artifacts/screening_robot_qwen3_1_7b_json")

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
lora_dir = ARTIFACT_ROOT / CONFIG["lora_dir"]
gguf_dir = ARTIFACT_ROOT / CONFIG["gguf_dir"]

model.save_pretrained(str(lora_dir))
tokenizer.save_pretrained(str(lora_dir))
print(f"LoRA adapter saved to: {lora_dir}")

In [ ]:
gguf_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained_gguf(
    str(gguf_dir),
    tokenizer,
    quantization_method="q4_k_m",
)

gguf_files = sorted(gguf_dir.rglob("*.gguf"))
assert gguf_files, "Expected at least one GGUF artifact after export."
for gguf_file in gguf_files:
    size_mb = gguf_file.stat().st_size / 1024**2
    print(f"{gguf_file} ({size_mb:.1f} MB)")

In [ ]:
del trainer, model
cleanup_memory()

%pip install -q llama-cpp-python

from llama_cpp import Llama

gguf_path = str(gguf_files[0])
print(f"Loading smoke-test GGUF: {gguf_path}")

llm = Llama(
    model_path=gguf_path,
    n_ctx=2048,
    n_threads=os.cpu_count() or 4,
    n_gpu_layers=-1,
    verbose=False,
)

smoke_response = cast(
    Any,
    llm.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": build_user_message(
                    "high fever, severe headache, myalgia, rash, nausea"
                ),
            },
        ],
        temperature=0.0,
        max_tokens=256,
    ),
)
smoke_text = str(smoke_response["choices"][0]["message"].get("content", "") or "")
print(smoke_text)

smoke_parsed = parse_prediction(smoke_text)
assert smoke_parsed["json_parsed"], "Smoke-test output must parse as JSON."
assert smoke_parsed["schema_valid"], "Smoke-test output must satisfy the structured schema."
display(pd.DataFrame([smoke_parsed]))